# Colab — Fine-tune VideoMAE-Small di Big Broiler (PBL Kesehatan Ayam, Stage 2 lanjutan)

Latihan **lanjutan** model perilaku Stage 2 (yang sudah dilatih di
ChickenAct, akurasi test-set 91,99%) memakai klip **video kandang
beneran Big Broiler** (091 top-view + 080 sudut miring + 075 sudut
miring). Tujuannya: model belajar seperti apa ayam makan/minum dari
atas dan dari miring — karena di lapangan, INGESTIVE ketukar RESTING
terus (recall INGESTIVE cuma 13,11% top-view / 28,77% miring).

## Akun — PENTING, baca dulu
- Training ini jalan di **Colab dengan akun `pandumanis72@gmail.com`**
  (akun yang sama seperti Stage 2). Saat cell mount Drive meminta
  login Google, **pilih akun `pandumanis72@gmail.com`**.
- Model awal = model Stage 2 (`MyDrive/PBL-Ayam/runs/stage2-videomae/best/`,
  sudah ada di Drive dari training 11 Nov) — BUKAN dari Kinetics lagi.
- Dataset = `BigBroiler-Finetune.zip` di `MyDrive/PBL-Ayam/dataset/` (sudah
  di-upload, 554.110.228 byte, md5 `28a3308ee3b307ac7c0552a3f095a4e3`).
- Checkpoint di-sync ke Drive (`MyDrive/PBL-Ayam/runs/stage2-bigbroiler-finetune/`)
  tiap 120 detik, pola yang sama seperti Stage 2. Tidak ada file ID
  Drive yang di-hardcode — semua path berupa nama folder/file.

## Dataset — isi zip (sudah disiapkan, tinggal pakai)
- **2.355 klip** (16 frame @224x224 per klip, crop mengikuti kotak GT):
  train 967 (INGESTIVE 435 / ACTIVE 97 / RESTING 435 — RESTING
  di-downsample ke jumlah INGESTIVE supaya latihnya seimbang),
  val 169, test 1.011 (paruh-2 video 091), test-075 208 (video 075
  seluruhnya, video yang belum pernah dilihat).
- Split: train/val dari **091 paruh-1 + 080**; test utama = **091
  paruh-2** (GT rapat, semua kelas ada); test kedua = **075**
  (generalisasi ke video baru — INGESTIVE-nya cuma 4 klip, jadi angka
  test-075 untuk INGESTIVE hampir tidak berarti, yang dilihat terutama
  ACTIVE/RESTING).
- Label = suara mayoritas aksi GT (pemetaan sama seperti
  `compare_gt.py`; Explor_Forag -> ACTIVE). Detail lengkap ada di
  `dataset_info.json` di dalam zip.
- **Catatan jujur:** belum ada yang tahu berapa naiknya akurasi
  setelah fine-tune ini — diukur lawan kedua test set setelah latih
  (lihat langkah terakhir di bawah). Kalau tidak naik, itu juga hasil.

## Alur notebook
1. Cek GPU + mount Drive — 2. Konfigurasi — 3. Dataset dari Drive —
   4. Unzip + cek manifest — 5. Clone repo + install dependensi —
   6. Verifikasi manifest + distribusi — 7. Salin model Stage 2 dari
   Drive — 8. Training (sync checkpoint ke Drive tiap 120 detik) —
   9. Baca metrik training.


In [ ]:
# 0. Cek GPU + mount Drive (login dengan akun pandumanis72@gmail.com)
import torch
print('torch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: GPU tidak terdeteksi. Aktifkan Runtime > Change runtime type > T4 GPU, lalu run ulang.')

from google.colab import drive
drive.mount('/content/drive')  # <-- pilih akun pandumanis72@gmail.com
print('Drive termount di /content/drive')


In [ ]:
# 1. Konfigurasi — ubah hanya di sini kalau perlu
EPOCHS = 4  # sedikit saja: latihan lanjutan, hindari catastrophic forgetting
RUN_NAME = 'stage2-bigbroiler-finetune'  # folder run BARU di Drive (jangan timpa stage2-videomae)
# Model awal = model Stage 2 hasil training 11 Nov (ada di Drive).
# BUKAN checkpoint Kinetics — backbone + head yang sudah dilatih dipertahankan.
STAGE2_MODEL_DRIVE = '/content/drive/MyDrive/PBL-Ayam/runs/stage2-videomae/best'
LR = 1e-5  # lebih kecil dari Stage 2 (5e-5) — penyesuaian halus, bukan latih ulang
TRAIN_BATCH = 16
EVAL_BATCH = 32
SEED = 42

# Dataset: HANYA dari Drive (zip sudah disiapkan + di-upload).
# Tidak ada unduhan Zenodo di notebook ini.
ZIP_NAME = 'BigBroiler-Finetune.zip'
EXPECTED_SIZE = 554110228
EXPECTED_MD5 = '28a3308ee3b307ac7c0552a3f095a4e3'

DRIVE_ROOT = '/content/drive/MyDrive/PBL-Ayam'
DRIVE_DATASET_DIR = f'{DRIVE_ROOT}/dataset'
DRIVE_RUNS_DIR = f'{DRIVE_ROOT}/runs'
LOCAL_ROOT = '/content/bigbroiler_work'
LOCAL_ZIP = f'{LOCAL_ROOT}/{ZIP_NAME}'
EXTRACT_DIR = f'{LOCAL_ROOT}/BigBroiler-Finetune'
FRAMES_DIR = f'{EXTRACT_DIR}/frames'  # zip berisi folder frames/ di top-level
LOCAL_MODEL = f'{LOCAL_ROOT}/stage2_model'
LOCAL_OUTPUT = f'/content/videomae_output/{RUN_NAME}'
DRIVE_RUN_DIR = f'{DRIVE_RUNS_DIR}/{RUN_NAME}'
REPO_DIR = '/content/chicken-anomaly-system'
STAGE2_DIR = f'{REPO_DIR}/backend/stage2_behavior'

import os
for d in (DRIVE_DATASET_DIR, DRIVE_RUNS_DIR, LOCAL_ROOT, DRIVE_RUN_DIR):
    os.makedirs(d, exist_ok=True)
print('ZIP_NAME     :', ZIP_NAME)
print('FRAMES_DIR   :', FRAMES_DIR)
print('LOCAL_MODEL  :', LOCAL_MODEL)
print('DRIVE_RUN_DIR:', DRIVE_RUN_DIR)
print('LOCAL_OUTPUT :', LOCAL_OUTPUT)


In [ ]:
# 2. Dataset — dari Drive (cari berdasarkan nama file saja).
#    Kalau zip belum ada di Drive, cell ini berhenti dengan pesan jelas.
import os, shutil, subprocess, hashlib

def md5_of_file(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

drive_zip = None
candidate = os.path.join(DRIVE_DATASET_DIR, ZIP_NAME)
if os.path.exists(candidate):
    drive_zip = candidate
else:
    print('Tidak ada di', DRIVE_DATASET_DIR, '- mencari di seluruh Drive ...')
    found = subprocess.run(
        ['find', '/content/drive/MyDrive', '-maxdepth', '4',
         '-name', ZIP_NAME],
        capture_output=True, text=True).stdout.splitlines()
    found = [p for p in found if os.path.exists(p)]
    if found:
        drive_zip = found[0]
        print('Ditemukan di Drive:', drive_zip)

assert drive_zip is not None, (
    f'{ZIP_NAME} tidak ditemukan di Drive akun ini. '
    'Pastikan file itu ada di MyDrive/PBL-Ayam/dataset/ (akun pandumanis72), '
    'lalu Run All lagi.')
print('Pakai zip dari Drive:', drive_zip, os.path.getsize(drive_zip), 'byte')
if not (os.path.exists(LOCAL_ZIP)
        and os.path.getsize(LOCAL_ZIP) == os.path.getsize(drive_zip)):
    print('Menyalin zip dari Drive ke sesi ini ...')
    shutil.copyfile(drive_zip, LOCAL_ZIP)
else:
    print('Salinan lokal sudah ada.')

print('Verifikasi zip:', LOCAL_ZIP, os.path.getsize(LOCAL_ZIP), 'byte')
assert os.path.getsize(LOCAL_ZIP) == EXPECTED_SIZE, (
    f"Ukuran zip {os.path.getsize(LOCAL_ZIP)} != {EXPECTED_SIZE}")
print('md5:', md5_of_file(LOCAL_ZIP))
assert md5_of_file(LOCAL_ZIP) == EXPECTED_MD5, 'md5 zip tidak cocok!'
print('Zip OK (ukuran + md5 cocok).')


In [ ]:
# 3. Unzip + cek manifest dataset
import os, zipfile, json
from collections import Counter

if not (os.path.isdir(FRAMES_DIR)
        and os.path.exists(os.path.join(FRAMES_DIR, 'frames_manifest.json'))):
    print('Ekstrak', LOCAL_ZIP, '->', EXTRACT_DIR, '...')
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(EXTRACT_DIR)
    print('Ekstrak selesai.')
else:
    print('Folder ekstrak sudah ada:', FRAMES_DIR)

manifest_path = os.path.join(FRAMES_DIR, 'frames_manifest.json')
with open(manifest_path) as f:
    entries = json.load(f)
print('Entri manifest utama:', len(entries))
print('Per split/kelas:')
for k, v in sorted(Counter((e['split'], e['label4']) for e in entries).items()):
    print(f'  {k[0]:6s} {k[1]:10s}: {v}')
assert len(entries) == 2147, 'Jumlah entri manifest tidak sesuai harapan (2147).'
assert {e['split'] for e in entries} == {'train', 'val', 'test'}, (
    'Split di manifest tidak lengkap (train/val/test).')
info_path = os.path.join(FRAMES_DIR, 'dataset_info.json')
with open(info_path) as f:
    info = json.load(f)
print('Test-075 (manifest terpisah):',
      info['counts']['total_test075_manifest'], 'klip')
print('Manifest OK.')


In [ ]:
# 4. Clone repo (branch stage2) + install dependensi training
import os, subprocess, sys

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo sudah ada, pull terbaru ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout',
                    'stage2/behavior-setup'], check=False)
else:
    subprocess.run(['git', 'clone', '--branch', 'stage2/behavior-setup',
                    'https://github.com/Ysoap/chicken-anomaly-system.git',
                    REPO_DIR], check=True)
    print('Repo ter-clone di', REPO_DIR)

print('Install transformers + accelerate ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.46,<5', 'accelerate'], check=True)  # >=4.46: warmup_ratio baru ada sejak 4.46; <5: checkpoint Stage 2 (4.x, q_bias/v_bias) harus termuat persis - 5.x menamai ulang jadi query.bias/value.bias dan me-reset 24 vektor bias
import transformers
print('transformers:', transformers.__version__)
print('Script Stage 2 ada:', os.path.exists(
    os.path.join(STAGE2_DIR, 'train.py')))


In [ ]:
# 5. Verifikasi manifest + distribusi (tidak ada data prep ulang —
#    klip sudah dipotong + diberi label di VM; di sini hanya dicek)
import os, json
from collections import Counter

with open(os.path.join(FRAMES_DIR, 'frames_manifest.json')) as f:
    entries = json.load(f)
with open(os.path.join(FRAMES_DIR, 'dataset_info.json')) as f:
    info = json.load(f)

print('Aturan label:', info['label_rule'])
print('Pemetaan dasar:', info['mapping_base'])
print('Perluasan pemetaan (aksi yang hanya ada di 080/075):',
      info['mapping_extension'])
print()
print('Distribusi train (sesudah balancing):',
      info['counts']['per_split_per_class']['train'])
print('Distribusi val:', info['counts']['per_split_per_class']['val'])
print('Distribusi test (091 paruh-2):',
      info['counts']['per_split_per_class']['test'])
print('Distribusi test-075:',
      info['counts']['per_split_per_class']['test075'])
print()
print('Cek file frame ada semua (sampel 200 entri pertama) ...')
missing = 0
for e in entries[:200]:
    for rel in e['frame_files']:
        if not os.path.exists(os.path.join(FRAMES_DIR, rel)):
            missing += 1
print('Frame hilang di sampel:', missing)
assert missing == 0, 'Ada file frame yang hilang — zip rusak, unduh ulang.'
print('Manifest + distribusi OK.')


In [ ]:
# 6. Salin model Stage 2 dari Drive (titik awal fine-tune)
import os, shutil, json

assert os.path.isdir(STAGE2_MODEL_DRIVE), (
    f'Model Stage 2 tidak ditemukan di Drive: {STAGE2_MODEL_DRIVE}. '
    'Folder best/ itu hasil training 11 Nov — pastikan masih ada di '
    'MyDrive/PBL-Ayam/runs/stage2-videomae/best/ (akun pandumanis72).')
if os.path.isdir(LOCAL_MODEL):
    shutil.rmtree(LOCAL_MODEL)
shutil.copytree(STAGE2_MODEL_DRIVE, LOCAL_MODEL)
print('Model Stage 2 tersalin ke:', LOCAL_MODEL)
for fn in ('config.json', 'model.safetensors'):
    p = os.path.join(LOCAL_MODEL, fn)
    assert os.path.exists(p), f'{fn} tidak ada di model Stage 2!'
    print(f'  {fn}: {os.path.getsize(p)} byte')
with open(os.path.join(LOCAL_MODEL, 'config.json')) as f:
    cfg = json.load(f)
print('num_labels:', cfg.get('num_labels'),
      '| id2label:', cfg.get('id2label'))
assert cfg.get('num_labels') == 3, 'Model awal harus 3 kelas.'
print('Model awal OK — training akan melanjutkan dari sini.')


## 7. Training — fine-tune lanjutan dari model Stage 2

Menjalankan `train.py` (HF Trainer) sebagai subprocess, dengan **sync
checkpoint ke Drive tiap 120 detik** (pola yang sama seperti Stage 2).
Yang di-sync: folder checkpoint terbaru (file model + state, TANPA
`optimizer.pt`/`scheduler.pt`/`rng_state.pth` supaya cepat —
konsekuensinya resume akan me-restart optimizer, dicatat di sini) dan
di akhir: folder `best/` + `train_metrics.json`.

**Yang beda dari Stage 2:** checkpoint awal = model Stage 2 (bukan
Kinetics), LR 1e-5 (lebih kecil), epoch 4 (sedikit — latihan lanjutan,
bukan latih ulang). Folder run baru: `runs/stage2-bigbroiler-finetune/`
— folder `stage2-videomae` yang lama TIDAK ditimpa.

**Resume:** kalau sesi Colab putus / limit habis, jalankan lagi notebook
ini dari awal (Run All) — cell training akan menemukan checkpoint di
Drive dan melanjutkan dari situ (bukan dari nol). Jangan hapus folder
run di Drive sebelum training selesai.

**Perkiraan waktu:** train 967 klip, batch 16 -> ~61 step/epoch,
4 epoch ~244 step. Di T4 kira-kira 15–30 menit (jauh lebih cepat dari
Stage 2 yang 2 jam, karena datanya jauh lebih sedikit).


In [ ]:
# 7. Training VideoMAE-Small (fine-tune lanjutan dari model Stage 2) + sync Drive
import os, subprocess, threading, time, shutil, sys, glob

os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)

# --- Resume: cari checkpoint terbaru di Drive (akun baru) ---
resume_arg = None
drive_ckpts = sorted(glob.glob(os.path.join(DRIVE_RUN_DIR, 'checkpoint-*')))
if drive_ckpts:
    latest_drive = drive_ckpts[-1]
    print('Ditemukan checkpoint di Drive, resume dari situ:', latest_drive)
    local_ckpt = os.path.join(LOCAL_OUTPUT, os.path.basename(latest_drive))
    if os.path.isdir(local_ckpt):
        shutil.rmtree(local_ckpt)
    shutil.copytree(latest_drive, local_ckpt)
    resume_arg = 'auto'
else:
    print('Tidak ada checkpoint di Drive, training dari awal '
          '(fine-tune dari model Stage 2).')

# --- Sync ke Drive selama training (cek tiap 120 detik) ---
_sync_stop = threading.Event()
_sync_seen = {}

def _copy_file_if_changed(src, dst):
    try:
        if not os.path.exists(src):
            return False
        key = (os.path.getsize(src), os.path.getmtime(src))
        if _sync_seen.get(src) == key:
            return False
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        tmp = dst + '.tmp'
        shutil.copyfile(src, tmp)
        os.replace(tmp, dst)
        _sync_seen[src] = key
        print(f'[sync Drive] {os.path.basename(src)} -> {dst} '
              f'({key[0]} byte)')
        return True
    except Exception as e:
        print(f'[sync Drive] gagal salin {src}: {e}')
        return False

# File model/state yang di-sync dari checkpoint (optimizer dkk dilewati).
_SYNC_FILES = ('config.json', 'model.safetensors', 'pytorch_model.bin',
               'trainer_state.json', 'training_args.bin',
               'generation_config.json', 'preprocessor_config.json')

def _sync_latest_checkpoint():
    ckpts = sorted(glob.glob(os.path.join(LOCAL_OUTPUT, 'checkpoint-*')))
    if not ckpts:
        return
    latest = ckpts[-1]
    dst_dir = os.path.join(DRIVE_RUN_DIR, os.path.basename(latest))
    os.makedirs(dst_dir, exist_ok=True)
    for fn in _SYNC_FILES:
        _copy_file_if_changed(os.path.join(latest, fn),
                              os.path.join(dst_dir, fn))

def _sync_loop():
    while not _sync_stop.is_set():
        _sync_latest_checkpoint()
        _sync_stop.wait(120)

_sync_thread = threading.Thread(target=_sync_loop, daemon=True)
_sync_thread.start()
print('Sync checkpoint ke Drive aktif (cek tiap 120 detik).')

cmd = [sys.executable, os.path.join(STAGE2_DIR, 'train.py'),
       '--frames-manifest', os.path.join(FRAMES_DIR, 'frames_manifest.json'),
       '--data-root', FRAMES_DIR,
       '--checkpoint', LOCAL_MODEL,
       '--output-dir', LOCAL_OUTPUT,
       '--epochs', str(EPOCHS), '--lr', str(LR),
       '--train-batch', str(TRAIN_BATCH), '--eval-batch', str(EVAL_BATCH),
       '--seed', str(SEED)]
if resume_arg:
    cmd += ['--resume', resume_arg]
    print('Mode: RESUME dari checkpoint Drive')
else:
    print('Mode: FINE-TUNE BARU dari model Stage 2')
print('Perintah:', ' '.join(cmd))

try:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print('Training process exit code:', proc.returncode)
finally:
    _sync_stop.set()
    _sync_thread.join(timeout=30)
    _sync_latest_checkpoint()
    # Sync hasil akhir: model terbaik + metrik.
    best_local = os.path.join(LOCAL_OUTPUT, 'best')
    if os.path.isdir(best_local):
        best_drive = os.path.join(DRIVE_RUN_DIR, 'best')
        shutil.copytree(best_local, best_drive, dirs_exist_ok=True)
        print('Model terbaik tersalin ke Drive:', best_drive)
    _copy_file_if_stable = _copy_file_if_changed
    _copy_file_if_stable(os.path.join(LOCAL_OUTPUT, 'train_metrics.json'),
                         os.path.join(DRIVE_RUN_DIR, 'train_metrics.json'))
    print('Sync checkpoint terakhir ke Drive selesai.')

if proc.returncode != 0:
    raise RuntimeError(
        f"Training GAGAL (exit code {proc.returncode}). "
        "Lihat error di output cell ini di atas — jangan lanjut, "
        "perbaiki dulu baru Run All lagi.")

print('Training selesai / sesi ini berakhir.')
print('Model terbaik lokal:', os.path.join(LOCAL_OUTPUT, 'best'))
print('Model terbaik Drive:', os.path.join(DRIVE_RUN_DIR, 'best'))


In [ ]:
# 8. Baca metrik training (akurasi val + per kelas)
import os, json

metrics_path = os.path.join(LOCAL_OUTPUT, 'train_metrics.json')
drive_metrics = os.path.join(DRIVE_RUN_DIR, 'train_metrics.json')
path = metrics_path if os.path.exists(metrics_path) else drive_metrics
print('Metrik:', path)
if os.path.exists(path):
    with open(path) as f:
        m = json.load(f)
    print('Checkpoint awal:', m.get('checkpoint'))
    print('Kelas dilatih:', m.get('classes'), '| num_labels:', m.get('num_labels'))
    print('Epoch:', m.get('epochs'), '| LR:', m.get('lr'))
    print('train_loss:', m.get('train_loss'))
    print('Metrik val (model terbaik):')
    for k, v in m.get('val_metrics_best', {}).items():
        print(f'  {k}: {v}')
else:
    print('train_metrics.json belum ada — training belum selesai / gagal.')
print()
print('Model terbaik lokal ada:', os.path.isdir(os.path.join(LOCAL_OUTPUT, 'best')))
print('Model terbaik Drive ada:', os.path.isdir(os.path.join(DRIVE_RUN_DIR, 'best')))


## Selesai untuk notebook ini — langkah berikutnya (bukan di notebook ini)

1. **Cek metrik di atas.** Akurasi val di sini dihitung pada val kecil
   (169 klip) — jangan jadikan patokan utama. Patokan yang jujur adalah
   evaluasi test set di bawah.
2. **Evaluasi test set** — dua test set, dua perintah (jalankan di VM
   asisten atau di Colab; di CPU kira-kira 40 menit untuk test utama +
   8 menit untuk test-075 pada 0,42 klip/detik):
   ```
   python backend/stage2_behavior/eval_test_set.py \
       --checkpoint <folder best hasil fine-tune ini> \
       --frames-manifest <FRAMES_DIR>/frames_manifest.json \
       --data-root <FRAMES_DIR> \
       --output results/bigbroiler_finetune_test_eval.json
   python backend/stage2_behavior/eval_test_set.py \
       --checkpoint <folder best hasil fine-tune ini> \
       --frames-manifest <FRAMES_DIR>/frames_manifest_test075.json \
       --data-root <FRAMES_DIR> \
       --output results/bigbroiler_finetune_test075_eval.json
   ```
   Bandingkan dengan baseline SEBELUM fine-tune (diukur lawan GT):
   test utama (091 paruh-2) ~61,8% top-view, test-075 belum pernah
   diukur. Kalau akurasi naik dan recall INGESTIVE naik, fine-tune
   berhasil; kalau tidak naik, laporkan apa adanya — itu juga hasil.
3. **Simpan model ke repo** (Git LFS, seperti model Stage 2):
   `backend/stage2_behavior/models/` — catat di README Stage 2.
4. Setelah itu baru: uji `inference.py` Stage 2 di video 080/091
   dengan model baru, lalu Stage 3 (aturan durasi ->
   aman/dicurigai/sakit + ambang).

Kredit dataset: Big Broiler Dataset, Cardoen dkk., UGent/DECIDE,
CV4Animals 2025, lisensi AGPL-3.0 (atribusi wajib di laporan).
Model awal: VideoMAE-Small hasil fine-tune Stage 2 (ChickenAct),
input 16 frame @ 224x224.
